# Caso Banca 360: Master Notebook Orquestador V3

Este notebook mantiene la secuencia del caso original, pero la v3 endurece la metodologia con parsimonia explicita, incertidumbre cuantificada, validacion temporal con purga y embargo, brecha de valor frente a consenso y auditoria de equidad para decisiones de retencion.

La narrativa sigue viviendo aqui; la logica estadistica, el score operativo, la parsimonia AIC/BIC, el bootstrap, la capa SHAP, el CLV y la activacion CRM residen en `src/banca_360_mlops/`.

In [1]:
from pathlib import Path

import sys

import pandas as pd

from IPython.display import display

# El notebook v3 solo resuelve rutas, carga configuracion y expone la interfaz narrativa.
# La seleccion de modelos, la parsimonia, la incertidumbre y la activacion viven en src/ para que notebook y CLI ejecuten exactamente la misma logica.
PROJECT_ROOT = Path.cwd()

while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():

    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:

    sys.path.insert(0, str(PROJECT_ROOT))

from src.banca_360_mlops.config import load_project_config

from src.banca_360_mlops.services.bank360_case import Bank360CaseService

config = load_project_config(PROJECT_ROOT)

service = Bank360CaseService(config)

contract_bundle = service.get_case_contract()

contract = contract_bundle['contract']

mapa_variables = contract_bundle['mapa_variables']

display(
    pd.DataFrame(
        [
            {'componente': 'project_root', 'valor': str(config.project_root)},
            {'componente': 'python_requires', 'valor': '>=3.11,<3.12'},
            {'componente': 'target', 'valor': contract['target']},
            {'componente': 'bootstrap_iterations', 'valor': config.case.uncertainty.bootstrap_iterations},
            {'componente': 'temporal_splits', 'valor': config.case.temporal_validation.n_splits},
        ]
    )
)

display(mapa_variables)

,componente,valor
0,project_root,C:\Users\yonai\Documents\Master Analytics and ...
1,python_requires,">=3.11,<3.12"
2,target,abandono
3,bootstrap_iterations,40
4,temporal_splits,4


,bloque,variables_clave,lectura_de_negocio
0,valor,"ingreso_mensual, saldo_promedio_3m, valor_clie...",Mide cuanto valor esta realmente en juego.
1,vinculacion,"productos_activos, nomina_domiciliada, hipotec...",Refleja profundidad de relacion con el banco.
2,comportamiento,"compras_12m, transacciones_app_30d, indice_eng...",Resume intensidad de uso y respuesta comercial.
3,friccion,"reclamaciones, satisfaccion, atraso_30d, saldo...",Captura senales de desgaste o dolor del cliente.
4,canal,"usa_app, canal_servicio, interacciones_sucursa...",Describe por donde conviene activar la retencion.
5,decision,"abandono, probabilidad_abandono, probabilidad_...",Convierte evidencia en priorizacion operativa ...


## Marco metodologico v3 y construccion del caso

La v3 sigue empezando por negocio, dato y trazabilidad, pero ahora aterriza cinco cambios conceptuales concretos: parsimonia con criterios de informacion, variables con memoria parcial mediante diferenciacion fraccional, lectura de incertidumbre por intervalos, validacion temporal conservadora y una capa formal de equidad.

WAIC se documenta como extension bayesiana opcional: solo es correcto cuando existe una posterior probabilistica del modelo. En este caso tabular frequentista se usa AIC/BIC donde son comparables y se deja WAIC como compuerta metodologica, no como numero decorativo.

In [2]:
# Este bloque construye el contexto metodologico y el dataset operativo del caso.
# La v3 agrega un resumen formal de la metodologia nueva y materializa variables con diferenciacion fraccional para preservar memoria temporal sin destruir completamente la senal historica.
context = service.build_context()
dataset = service.build_dataset()
df_bank = dataset['data']

display(context['manual_resumen'])
display(context['guia_metricas'].head(10))
display(context['referencia_frameworks'].head(10))
display(context['metodologia_v3_resumen'])
display(context['alineacion_metodologica_pdf'])
display(df_bank.head())
display(dataset['auditoria_inicial']['summary'])
display(dataset['tabular_standards']['summary'])
display(dataset['sampling_audit']['sampling_plan'])
display(dataset['data_dictionary']['dictionary'].head(15))
display(dataset['distribucion_objetivo'])
display(dataset['resumen_numerico'].head(15))
display(dataset['resumen_categorico'].head(12))
display(pd.Series(dataset['summary'], name='valor').to_frame())

,fase,objetivo,estandares
0,Ingesta y auditoria,"Validar integridad, trazabilidad y representat...",CRISP-DM - Business Understanding y Data Under...
1,ETL y saneamiento,"Estandarizar el dato, controlar outliers y dej...",CRISP-DM - Data Preparation | DAMA-DMBOK - Dat...
2,EDA y validacion de supuestos,"Entender distribuciones, relaciones y riesgos ...",CRISP-DM - Data Understanding | DAMA-DMBOK - D...
3,Modelado,Construir baselines reproducibles e interpreta...,CRISP-DM - Modeling | DAMA-DMBOK - Analytics G...
4,Inferencia,Cuantificar diferencias y relaciones de forma ...,CRISP-DM - Evaluation | DAMA-DMBOK - Data Scie...
5,Conclusiones y accion,"Traducir hallazgos tecnicos a decisiones, ries...",CRISP-DM - Deployment | DAMA-DMBOK - Data Gove...


,metrica,lectura_ejecutiva,accion_bi
0,p_value,Cuanta evidencia hay contra una explicacion po...,"Si es bajo y el efecto es material, prioriza l..."
1,r2,Porcentaje de variabilidad del negocio explica...,"Si es alto, el modelo sirve para planificacion..."
2,mae,Error medio absoluto en la misma unidad del KP...,Compuralo contra la tolerancia del negocio par...
3,rmse,Error que penaliza mas los fallos grandes y an...,"Si supera la tolerancia financiera, ajusta fea..."
4,smape,Error porcentual simetrico util para comparar ...,Usalo como apoyo explicativo cuando necesitas ...
5,mae_relativo_baseline,Cuanto mejora o empeora el error medio frente ...,"Si es menor que 1, el modelo supera al baselin..."
6,roc_auc,Capacidad de ordenar correctamente casos de ma...,"Usalo para priorizacion comercial, antifraude ..."
7,f1,"Balance entre precision y recall, util cuando ...","Si es bajo, revisa coste de falsos positivos y..."
8,vif,Nivel de redundancia entre predictores; valore...,Reduce variables redundantes o regulariza si n...


,metodologia,fases_principales,fortalezas_tecnicas,aplicabilidad_2025
0,CRISP-DM,"Negocio, Datos, Preparacion, Modelado, Evaluac...",Alineacion estrategica end-to-end y trazabilid...,"Estandar de oro para proyectos corporativos, c..."
1,SEMMA,"Sample, Explore, Modify, Model, Assess","Profundiza en exploracion, transformacion y se...",Muy util en laboratorios analiticos y experime...
2,KDD,"Seleccion, Limpieza, Transformacion, Mineria, ...",Enfatiza descubrimiento de patrones y extracci...,Adecuado para investigacion aplicada y mineria...
3,TDSP,"Negocio, Adquisicion, Modelado, Despliegue, Ac...","Escalabilidad cloud, colaboracion y disciplina...","Preferido en entornos agiles con CI/CD, observ..."


,pilar,que_cambia,operativizacion
0,Parsimonia,La seleccion deja de depender solo del mejor R...,"Benchmark logit anidado con AIC/BIC, pesos de ..."
1,Incertidumbre,El score deja de leerse como un numero unico y...,Bootstrap multivariable sobre el bloque fuera ...
2,Temporalidad,El holdout temporal se blinda frente a leakage...,Validacion cronologica con purga previa y emba...
3,Consenso y operacion,La accion comercial mide si la senal supera al...,Brecha de score/valor frente a consenso y poli...
4,Equidad y DL,La metodologia declara controles eticos y comp...,Fairness audit por grupo y checklist de early ...


,principio_pdf,aterrizaje_v2,componente
0,Muestreo probabilistico y estratificado,"Auditoria de representatividad por segmento, a...",audit_sampling_representativeness
1,"FAIR, metadata y trazabilidad",Diccionario de datos procesable con roles sema...,build_dataset_data_dictionary
2,Calidad tabular y reglas CSV reutilizables,"Contrato tabular con cabeceras normalizadas, t...",audit_dataset.tabular_standards
3,EDA iterativo con tipado correcto y trazabilid...,Se mantiene la auditoria estadistica existente...,build_dataset + run_methodology_validation
4,Parsimonia con penalizacion explicita,"Familia logistica anidada con AIC, BIC y pesos...",run_logistic_parsimony_study
5,Incertidumbre y prediccion por intervalos,Bootstrap multivariable sobre el holdout tempo...,run_bootstrap_prediction_intervals
6,Integridad temporal con purga y embargo,Ventanas cronologicas que excluyen observacion...,run_purged_temporal_validation
7,Gobernanza etica y equidad,"Fairness audit con paridad de seleccion, equal...",run_fairness_audit


...

,columna,columna_estandar,rol,familia_semantica,tipo_dato,descripcion,pct_nulos,valores_unicos,pct_unicidad,nullable,ejemplo_valores
0,registro_id,registro_id,identifier,numeric,int64,Identificador tecnico de registro generado par...,0.00,1200,100.00,False,1 | 2 | 3
1,cliente_id,cliente_id,identifier,numeric,int64,Identificador unico de cliente visible para sc...,0.00,1200,100.00,False,1 | 2 | 3
2,fecha_registro,fecha_registro,timestamp,datetime,datetime64[ns],Fecha original de alta o captura historica del...,0.00,583,48.58,False,2024-03-06 | 2025-07-18 | 2025-04-22
3,fecha_corte,fecha_corte,timestamp,datetime,datetime64[ns],Fecha de corte analitico usada para auditoria ...,0.00,490,40.83,False,2026-02-11 | 2025-02-07 | 2025-04-12
4,region,region,feature,categorical,object,Region geografica del cliente para revisar cob...,0.00,5,0.42,False,Centro | Este | Oeste
5,canal_captacion,canal_captacion,feature,categorical,object,Canal principal de adquisicion del cliente.,0.00,4,0.33,False,Tienda | Tienda | Web
6,canal_servicio,canal_servicio,feature,categorical,object,Canal dominante de atencion o relacion operati...,0.00,3,0.25,False,Digital | Digital | Mixto
7,segmento,segmento,feature,categorical,object,Segmento comercial base del cliente.,0.00,3,0.25,False,Premium | Estandar | Basico
8,macro_segmento,macro_segmento,feature,categorical,object,Agrupacion estrategica del cliente para priori...,0.00,4,0.33,False,Valor | Masivo | Masivo
9,edad,edad,feature,numeric,float64,Edad del cliente en anos.,0.00,55,4.58,False,35.0 | 48.0 | 18.0


,abandono,pct_clientes
0,0,67.75
1,1,32.25


,variable,count,mean,std,min,25%,50%,75%,max
0,registro_id,1200.0,600.500000,346.554469,1.0000,300.7500,600.5000,900.25000,1200.0000
1,cliente_id,1200.0,600.500000,346.554469,1.0000,300.7500,600.5000,900.25000,1200.0000
2,edad,1200.0,38.912500,10.771948,18.0000,31.0000,39.0000,46.00000,77.0000
3,antiguedad_meses,1200.0,60.130833,34.859284,1.0000,30.0000,58.0000,90.00000,120.0000
4,ingreso_mensual,1152.0,2412.903082,891.976995,475.2400,1792.2000,2233.8350,2880.10750,6787.6200
5,saldo_promedio_3m,1200.0,10809.166575,6016.125720,1519.3000,6560.3400,9460.9950,13608.68500,48268.9400
6,saldo_variacion_pct_3m,1165.0,0.057772,0.111413,-0.3261,-0.0174,0.0594,0.13190,0.4003
7,gasto_mensual,1200.0,927.303246,647.014174,176.9600,560.1450,793.3300,1116.72000,8883.2700
8,ticket_medio,1160.0,213.362724,225.946762,24.9500,98.5050,154.4950,245.40250,3701.7000
9,productos_activos,1200.0,3.117500,1.567080,1.0000,2.0000,3.0000,4.00000,7.0000


,variable,n_categorias,moda
0,fecha_registro,583,2024-04-12 00:00:00
1,fecha_corte,490,2026-01-15 00:00:00
2,region,5,Centro
3,canal_captacion,4,Web
4,canal_servicio,3,Digital
5,segmento,3,Estandar
6,macro_segmento,4,Masivo
7,usa_app,2,Si
8,nomina_domiciliada,2,No
9,hipoteca_activa,2,No


,valor
filas,1200
columnas,42
pct_abandono,32.25
max_missing_pct,87.0
leakage_count,0
tabular_alerts,0
sampling_alerts,0
fracdiff_features,3
interpretation,No se detectaron filas completamente duplicada...


## Benchmark con parsimonia y capa BI ejecutiva

La v3 ya no promueve un champion solo por liderar una metrica puntual. Primero compara algoritmos base, luego abre una familia logistica anidada donde AIC y BIC si son comparables, y finalmente aplica una politica Occam: si el gap de rendimiento es pequeno, sube una especificacion mas simple.

Sobre esa decision se monta la capa BI reusable para traducir el score a una lectura ejecutiva consistente, trazable y defendible ante negocio.

In [3]:
# El benchmark v3 combina rendimiento predictivo con parsimonia dentro de una familia comparable.
# Si una especificacion logistica mas simple queda suficientemente cerca del mejor ROC AUC generalista, la politica Occam la promueve y el resto del pipeline hereda ese subconjunto operativo de variables.
benchmark = service.run_benchmark(df_bank)
bi_layer = service.run_bi_layer(df_bank, benchmark)

display(benchmark['benchmark_df'])
display(benchmark['parsimonia']['summary'])
display(benchmark['parsimonia']['ensemble_metrics'])
display(bi_layer['bi_result']['modelado']['model']['metrics'])
display(bi_layer['bi_result']['modelado']['metric_translation'])
display(bi_layer['bi_result']['inferencia']['summary'])
display(bi_layer['bi_result']['conclusiones']['summary'])
display(
    pd.Series(
        {
            'modelo_champion_v3': benchmark['model_name'],
            'seleccion_occam': benchmark['selection_policy'],
            'n_features_operativas': len(benchmark['selected_features']),
            'gap_roc_auc_top2': benchmark['lead_gap_roc_auc'],
            'vif_max_bi': bi_layer['vif_max'],
        },
        name='valor',
    ).to_frame()
 )

,modelo,accuracy,precision,recall,f1,roc_auc,log_loss,complexity_rank,selected_for_v3
0,logistic,0.7267,0.7213,0.7267,0.7234,0.7642,0.5446,1,True
1,gradient_boosting,0.7167,0.7068,0.7167,0.7098,0.7505,0.5459,3,False
2,random_forest,0.7267,0.7110,0.7267,0.7067,0.7451,0.5440,4,False


,candidate,n_features,selected_features,roc_auc,log_loss,brier_score,aic,bic,log_likelihood,n_parameters,waic_status,akaike_weight
0,logistic_top_8,8,"edad, antiguedad_meses, ingreso_mensual, ingre...",0.7355,0.5589,0.1871,1005.7510,1048.9725,-493.8755,9,not_applicable_without_bayesian_posterior,0.000000
1,logistic_top_18,18,"edad, antiguedad_meses, ingreso_mensual, ingre...",0.7629,0.5417,0.1777,971.1156,1062.3611,-466.5578,19,not_applicable_without_bayesian_posterior,0.301724
2,logistic_top_12,12,"edad, antiguedad_meses, ingreso_mensual, ingre...",0.7371,0.5593,0.1866,1004.3654,1066.7965,-489.1827,13,not_applicable_without_bayesian_posterior,0.000000
3,logistic_top_25,25,"edad, antiguedad_meses, ingreso_mensual, ingre...",0.7629,0.5437,0.1807,969.4374,1113.5093,-454.7187,30,not_applicable_without_bayesian_posterior,0.698276
4,logistic_top_4,4,"edad, antiguedad_meses, ingreso_mensual, ingre...",0.6479,0.5954,0.2046,1091.3305,1115.3424,-540.6652,5,not_applicable_without_bayesian_posterior,0.000000


,roc_auc,log_loss,brier_score,waic_status
0,0.7646,0.5405,0.1788,not_applicable_without_bayesian_posterior


,accuracy,precision,recall,f1,roc_auc,log_loss
0,0.7,0.6873,0.7,0.6912,0.7575,0.5377


,metrica,valor,lectura_ejecutiva
0,accuracy,0.7000,"Interpreta la metrica junto a coste del error,..."
1,precision,0.6873,"Interpreta la metrica junto a coste del error,..."
2,recall,0.7000,"Interpreta la metrica junto a coste del error,..."
3,f1,0.6912,"El baseline es util, pero requiere revisar umb..."
4,roc_auc,0.7575,"Hay senal util para priorizacion, aunque todav..."
5,log_loss,0.5377,"Interpreta la metrica junto a coste del error,..."


,componente,metrica_clave,valor,lectura_ejecutiva
0,contraste_grupos,p_value,2.650574e-38,La evidencia estadistica es muy fuerte; pasa a...
1,modelo_ols,r2,9.989000e-01,El modelo explica gran parte de la variabilida...


,fase,hallazgo_clave,impacto_bi,accion_recomendada
0,Ingesta,Columnas con nulos relevantes,Los KPIs pueden estar sesgados si se imputan s...,Priorizar analisis MCAR/MAR/MNAR y documentar ...
1,EDA,Multicolinealidad relevante,Los coeficientes pueden volverse inestables y ...,"Reducir redundancias, agrupar features o usar ..."
2,Modelado,ROC AUC = 0.7575,"Hay senal util para priorizacion, aunque todav...","Usar el score para priorizar intervenciones, n..."
3,Inferencia,Kruskal-Wallis con p-value = 0.0000,La evidencia estadistica es muy fuerte; pasa a...,Combinar con tamano del efecto y visualizacion...
4,Gobierno continuo,"El pipeline ya separa ETL, EDA, modelado e inf...","La analitica se vuelve repetible, auditable y ...","Monitorear drift, recalibracion, calidad de da..."


,valor
modelo_champion_v3,logistic
seleccion_occam,Se mantiene logistic por liderazgo en ROC AUC ...
n_features_operativas,25
gap_roc_auc_top2,0.0137
vif_max_bi,8.2873


## Validacion avanzada, incertidumbre y equidad

La validacion v3 separa tres preguntas que antes podian mezclarse: si el modelo generaliza en tiempo sin fuga serial, si la prediccion puntual es estable o fragil bajo remuestreo y si el sistema reparte calidad de decision de forma desigual entre grupos sensibles.

La salida incluye validacion temporal purgada con embargo, intervalos de pronostico por bootstrap, fairness audit por edad y region, y un checklist explicito de controles que deberian exigirse si una red neuronal pasara a produccion.

In [4]:
# La validacion rigurosa v3 encapsula el bloque metodologico mas pesado del notebook.
# Aqui se auditan salud del pipeline, validacion temporal con purga y embargo, bootstrap de pronostico, fairness por grupo y la gobernanza minima exigible para deep learning antes de mover una red neuronal a produccion.
methodology = service.run_methodology_validation(df_bank, benchmark)
shap_result = service.run_shap_transparency(bi_layer['bi_result'])

display(methodology['execution_gate'])
display(methodology['temporal_validation']['summary'])
if not methodology['temporal_validation']['fold_report'].empty:
    display(methodology['temporal_validation']['fold_report'])
display(methodology['uncertainty']['portfolio_summary'])
if not methodology['uncertainty']['prediction_intervals'].empty:
    display(methodology['uncertainty']['prediction_intervals'].head(15))
display(methodology['fairness_audit']['summary'])
if not methodology['fairness_audit']['group_metrics'].empty:
    display(methodology['fairness_audit']['group_metrics'])
display(methodology['deep_learning_governance']['checklist'])
display(methodology['governance_audit']['tabular_standards']['summary'])
display(methodology['governance_audit']['sampling_audit']['sampling_plan'])
if not methodology['governance_audit']['sampling_audit']['alerts'].empty:
    display(methodology['governance_audit']['sampling_audit']['alerts'])
display(methodology['missingness_audit']['little_mcar'])
display(methodology['drift_report']['summary'])
display(methodology['rfe_report']['ranking'].head(10))
display(methodology['vif_report']['report'].head(10))
display(methodology['calibration_result']['metrics'])
if not methodology['calibration_result']['comparison'].empty:
    display(methodology['calibration_result']['comparison'])
display(methodology['risk_flags'])
if not bool(methodology['execution_gate'].iloc[0]['modeling_enabled']):
    print(methodology['execution_gate'].iloc[0]['accion_recomendada'])
if shap_result['available']:
    display(shap_result['summary'].head(12))
    display(shap_result['consistency'])
    display(shap_result['local_feature_table'].head(12))
else:
    print(shap_result['message'])

display(pd.Series(methodology['summary'], name='valor').to_frame())

El framework universal combina CRISP-DM, SEMMA, KDD y TDSP con una capa explicita de rigor estadistico, visual analytics, auditoria estructural de normalidad, diagnostico de dispersion residual y observabilidad para pipelines analiticos modernos.
No se detectaron filas completamente duplicadas en esta vista del dataset. Hay columnas con nulos relevantes. Antes de imputar, valida si el mecanismo parece MCAR, MAR o MNAR. No aparecieron candidatos de leakage evidentes con las reglas automaticas aplicadas.
En el subconjunto auditado de 26 columnas, la auditoria sugiere que los faltantes no son plenamente MCAR. Conviene priorizar KNN o MICE/Iterative Imputation y documentar el mecanismo como MAR o potencialmente MNAR.
Hay drift distribucional relevante en variables clave; valida cambio de captura, mezcla poblacional o estacionalidad antes de reutilizar el modelo.
Salud del pipeline 'caso_negocio_tabular': frescura = 170.47h, tasa de error de validacion = 9.83%. Decision operativa = degraded

,dataset,pipeline_health_profile,severidad_global,decision_operativa,alertas_activas,issues_criticos,modeling_enabled,accion_recomendada
0,caso_banca_360,demo,alerta,degraded,frescura,0,True,Permitir ejecucion degradada solo para diagnos...


,n_folds,mean_roc_auc,std_roc_auc,mean_log_loss,mean_brier_score
0,3,0.74,0.0315,0.572,0.1894


,fold,train_end,test_start,test_end,embargo_end,n_train,n_test,purged_rows,embargoed_rows,roc_auc,log_loss,brier_score
0,2,2025-02-14,2025-02-22,2025-07-04,2025-07-18,297,304,12,33,0.6956,0.6543,0.2117
1,3,2025-06-27,2025-07-05,2025-11-15,2025-11-29,594,288,19,34,0.7591,0.5410,0.1800
2,4,2025-11-08,2025-11-16,2026-03-29,2026-04-12,893,299,8,0,0.7654,0.5208,0.1764


,bootstrap_iterations_effective,mean_probability_base,mean_interval_width,high_risk_share_p05,high_risk_share_p95
0,40,0.2876,0.2466,0.0574,0.3684


...

,dimension,estado,implicacion
0,Missingness,Alerta,Las ausencias no lucen plenamente MCAR; convie...
1,Normalidad estructural,Seguimiento,"Aplicar Box-Cox o log, revisar colas con Ander..."
2,Parsimonia,Seguimiento,Se mantiene logistic por liderazgo en ROC AUC ...
3,Drift,Alerta,Hay columnas que exigen revision antes de esca...
4,Salud pipeline,Alerta,Permitir ejecucion degradada solo para diagnos...
5,Validacion temporal purgada,OK,La validacion temporal v3 entrena solo con his...
6,Calibracion,Seguimiento,"El score ordena bien, pero la probabilidad aun..."
7,Incertidumbre,OK,La v3 reporta intervalos de pronostico y no so...
8,Explicabilidad,Seguimiento,Conviene revisar la lectura causal agregada an...
9,Contrato tabular,OK,El dataset cumple razonablemente las reglas ta...


,feature,mean_abs_shap,std_abs_shap,mean_contribution
0,numeric__saldo_promedio_3m,0.769705,0.577230,0.090421
1,numeric__satisfaccion,0.304954,0.232017,-0.031085
2,numeric__reclamaciones,0.288870,0.066004,0.003010
3,categorical__usa_app_Si,0.262474,0.088627,-0.006385
4,numeric__productos_activos,0.237385,0.149246,-0.025972
5,numeric__transacciones_app_30d,0.219532,0.162676,0.023131
6,numeric__edad,0.189898,0.131316,0.004372
7,numeric__antiguedad_meses,0.140575,0.083797,0.012884
8,categorical__nomina_domiciliada_Si,0.124042,0.041892,0.001851
9,categorical__hipoteca_activa_Si,0.123269,0.060268,0.008127


,espacio_salida,error_abs_medio,error_abs_max,reconstruccion_consistente,n_observaciones_auditadas
0,decision_function,1.864249e-16,8.881784e-16,True,240


,feature,shap_value,abs_shap_value,valor_transformado,direccion_score
4,numeric__saldo_promedio_3m,2.352539,2.352539,-1.732248,Empuja al alza el score de abandono
11,numeric__transacciones_app_30d,0.462943,0.462943,1.044782,Empuja al alza el score de abandono
9,numeric__productos_activos,0.416049,0.416049,-1.283904,Empuja al alza el score de abandono
16,numeric__satisfaccion,0.404142,0.404142,-0.864723,Empuja al alza el score de abandono
29,categorical__macro_segmento_Recuperacion,-0.390012,0.390012,1.000000,Reduce el score de abandono
0,numeric__edad,0.356908,0.356908,-1.213428,Empuja al alza el score de abandono
17,numeric__reclamaciones,-0.252230,0.252230,-0.000000,Reduce el score de abandono
14,numeric__promociones_12m,0.236825,0.236825,-1.998502,Empuja al alza el score de abandono
1,numeric__antiguedad_meses,-0.205178,0.205178,0.720083,Reduce el score de abandono
20,categorical__usa_app_Si,-0.200392,0.200392,1.000000,Reduce el score de abandono


,valor
brier_score,0.1833
top_vif,10.2426
top_condition_index,7.3643
critical_belsley_count,0
simpson_detected,False
drift_columns,1
tabular_alerts,0
sampling_alerts,0
pipeline_health_profile,demo
pipeline_health_decision,degraded


## Activacion economica, consenso y politica bandit

La v3 sigue cerrando en dinero y accion, no solo en score. La diferencia es que ahora la salida operacional tambien mide cuanta senal nueva aporta el modelo frente al consenso del segmento y deja una politica de exploracion-explotacion lista para cuando entren recompensas reales por accion comercial.

Esto convierte la recomendacion de retencion en una pieza mas gobernable: umbral economico, shortlist, brecha de valor y primer marco para aprender operando.

In [5]:
# Este bloque traduce el score a decision economica y salida accionable de negocio.
# Ademas de CLV, umbral y segmentacion, la v3 reporta brecha contra consenso y una politica bandit inicial para no dejar la activacion comercial congelada en una prueba A/B estatica.
activation = service.run_clv_activation(df_bank, bi_layer['bi_result'])
artifacts = {
    'context': context,
    'dataset': dataset,
    'benchmark': benchmark,
    'bi_layer': bi_layer,
    'methodology': methodology,
    'shap': shap_result,
    'activation': activation,
}
execution_summary = service.build_execution_summary(artifacts)

display(activation['clv_result']['model_summary'])
display(activation['value_benchmark']['summary'])
display(activation['threshold_result']['recommended_threshold'])
display(activation['scorecard_result']['summary'])
display(activation['scorecard_result']['shortlist'].head(10))
display(activation['consensus_gap']['summary'])
display(activation['bandit_policy']['policy_table'])
display(activation['perfil_segmentos'])
display(activation['playbook_segmentos'])
display(pd.Series(execution_summary, name='valor').to_frame())

,motor,clientes_validos,clientes_repetidores,probabilidad_activo_media,clv_medio_12m,ticket_esperado_medio
0,lifetimes_bgnbd_gammagamma,1200,1200,0.995,1012.12,282.1


,algoritmo,modelo,mae,rmse,smape,mae_relativo_baseline,ratio_rmse_mae,mae_normalizado,rmse_normalizado,score_champion,es_champion
0,random_forest,Random Forest,226.0050,340.6398,8.6682,0.2380,1.5072,0.0,0.0,0.0,True
1,neural_network,Neural Network,271.4181,384.3252,11.2261,0.2858,1.4160,1.0,1.0,2.8,False


,umbral,clientes_contactados,pct_base_contactada,precision,recall,f1,churners_detectados,pct_churners_capturados,valor_recuperado_estimado,coste_campana,valor_esperado_neto,roi_estimado
0,0.5,62,25.83,0.5323,0.4286,0.4748,33,42.86,5187.01,1364.0,3823.01,2.8028


,riesgo_recomendado,macro_segmento,clientes,score_medio,valor_medio,valor_futuro_medio,valor_esperado_medio
2,Alto,Recuperacion,9,0.6155,550.15,550.15,26.79
0,Alto,Masivo,29,0.5766,617.12,617.12,30.61
1,Alto,Premium,4,0.5732,421.59,421.59,14.19
3,Alto,Valor,13,0.5643,1002.16,1002.16,58.63
6,Bajo,Recuperacion,13,0.2714,874.59,874.59,0.00
4,Bajo,Masivo,58,0.2281,1007.86,1007.86,0.00
7,Bajo,Valor,57,0.2016,1013.84,1013.84,0.00
5,Bajo,Premium,36,0.1896,1339.25,1339.25,0.00
9,Critico,Recuperacion,2,0.9139,496.32,496.32,42.13
8,Critico,Masivo,2,0.8333,570.48,570.48,48.65


,cliente_id,macro_segmento,clv_probabilistico_12m,canal_servicio,nomina_domiciliada,producto_premium,hipoteca_activa,usa_app,clv_probabilistico_6m,probabilidad_activo_clv,...,prioridad_integrada,riesgo_recomendado,prioridad_retencion,valor_futuro_probabilistico,valor_esperado_contacto,canal_recomendado,accion_recomendada,next_best_offer,brecha_score_vs_consenso,brecha_valor_vs_consenso
0,1122,Valor,4037.27,Digital,No,No,No,Si,2142.82,0.995,...,88.05,Alto,Alta,4037.27,293.78,Campana digital,Ofrecer vinculacion de nomina y paquete de fid...,Paquete nomina + cashback de captacion inmediata,0.2297,282.15
1,282,Valor,3311.59,Mixto,No,No,Si,No,1757.66,0.995,...,87.70,Alto,Alta,3311.59,237.02,Llamada consultiva,Ofrecer vinculacion de nomina y paquete de fid...,Incentivo de migracion digital y activacion de...,0.2297,225.39
2,731,Recuperacion,1336.85,Mixto,No,No,No,No,709.55,0.995,...,88.03,Alto,Alta,1336.85,102.17,Llamada consultiva,Ofrecer vinculacion de nomina y paquete de fid...,Incentivo de migracion digital y activacion de...,0.1702,90.12
3,151,Recuperacion,1433.80,Digital,No,No,No,Si,761.01,0.995,...,82.97,Alto,Alta,1433.80,90.15,Campana digital,Ofrecer vinculacion de nomina y paquete de fid...,Paquete nomina + cashback de captacion inmediata,0.0715,78.10
4,56,Masivo,1426.83,Sucursal,Si,Si,No,No,757.31,0.995,...,82.62,Alto,Alta,1426.83,89.60,Gestor especializado,Activar campana de retencion estandar con ince...,Incentivo de migracion digital y activacion de...,0.1643,79.23
5,455,Masivo,1142.95,Digital,No,No,No,No,606.63,0.995,...,84.00,Alto,Alta,1142.95,84.16,Campana digital,Ofrecer vinculacion de nomina y paquete de fid...,Incentivo de migracion digital y activacion de...,0.2630,73.79
6,1165,Masivo,777.79,Digital,No,No,No,Si,412.82,0.995,...,79.85,Critico,Alta,777.79,74.32,Campana digital,Ofrecer vinculacion de nomina y paquete de fid...,Monitoreo sin oferta economica inmediata,0.4714,63.95
7,1003,Masivo,1033.57,Digital,No,No,Si,No,548.58,0.995,...,82.43,Alto,Alta,1033.57,74.00,Campana digital,Ofrecer vinculacion de nomina y paquete de fid...,Incentivo de migracion digital y activacion de...,0.2630,63.63
8,1128,Masivo,1188.13,Digital,No,No,No,No,630.61,0.995,...,78.77,Alto,Alta,1188.13,70.93,Campana digital,Ofrecer vinculacion de nomina y paquete de fid...,Incentivo de migracion digital y activacion de...,0.1643,60.56
9,379,Recuperacion,753.01,Digital,No,No,No,No,399.67,0.995,...,77.55,Critico,Alta,753.01,70.65,Campana digital,Ofrecer vinculacion de nomina y paquete de fid...,Monitoreo sin oferta economica inmediata,0.3731,58.60


,macro_segmento,clientes,brecha_score_media,brecha_valor_media,score_medio,valor_medio
0,Masivo,95,0.0,-0.0016,0.3620,10.3684
1,Premium,43,-0.0,-0.0002,0.2443,1.3198
2,Recuperacion,27,0.0,0.0015,0.4548,12.0515
3,Valor,75,0.0,0.0040,0.2967,11.6340


,next_best_offer,n,mean_reward,median_reward,positive_reward_rate,alpha,beta,thompson_posterior_mean,epsilon_greedy_score,thompson_score
0,Incentivo de migracion digital y activacion de...,24,44.252500,20.57,1.000000,25,1,0.961538,0.9200,0.9615
1,Paquete nomina + cashback de captacion inmediata,25,32.215600,17.76,0.920000,24,3,0.888889,0.6752,0.6471
2,Oferta tactica de permanencia con cashback o p...,6,13.385000,13.73,1.000000,7,1,0.875000,0.2922,0.2647
3,Cross-sell de producto premium ligero sin cost...,3,0.000000,0.00,0.000000,1,4,0.200000,0.0200,0.0100
4,Monitoreo sin oferta economica inmediata,182,1.604011,0.00,0.038462,8,176,0.043478,0.0526,0.0022


,segmento_kmeans,cliente_id,probabilidad_abandono,probabilidad_activo_clv,valor_esperado_contacto,clv_probabilistico_6m,prioridad_integrada,segmento_nombre
0,0,235,0.333,0.995,7.272,461.160,49.895,Riesgo alto monetizable
1,1,5,0.234,0.995,106.160,2372.234,64.922,Riesgo critico de alto CLV


,segmento_nombre,clientes,pct_priorizados,score_medio,clv_6m_medio,valor_esperado_medio,oferta_dominante,bandit_arm_recomendado
0,Riesgo alto monetizable,235,0.255,0.333,461.160,7.272,Monitoreo sin oferta economica inmediata,Incentivo de migracion digital y activacion de...
1,Riesgo critico de alto CLV,5,0.400,0.234,2372.234,106.160,Monitoreo sin oferta economica inmediata,Incentivo de migracion digital y activacion de...


,valor
champion_model,logistic
benchmark_roc_auc,0.7642
benchmark_f1,0.7234
bi_roc_auc,0.7575
bi_log_loss,0.5377
brier_score,0.1833
drift_columns,1
pipeline_health_profile,demo
pipeline_health_decision,degraded
selected_feature_count,25


## Cierre operativo

El notebook sigue actuando como interfaz narrativa, pero la v3 deja persistidos artefactos nuevos de alto valor metodologico: benchmark de parsimonia, validacion temporal purgada, intervalos de pronostico, fairness audit, checklist de gobierno DL, brecha de valor contra consenso y politica bandit, junto con las salidas tradicionales del pipeline y el tracking en MLflow.